# Cascade

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/grounded-ai/grounded_ai/blob/main/examples/cascade.ipynb)

An LLM judge is accurate but slow and expensive. A decision model (Jev, OpenAI's Decisions API, or any model served at `/v1/systemone`, such as our open [agent-decision](https://github.com/grounded-ai/agent-decision)) answers in one cheap request and reports how sure it is, measured from the probabilities rather than self-reported.

`CascadeEvaluator` puts the two together: the decision model answers every question, and only the answers below `min_confidence` go to the judge. On 200 real agent turns this cut the bill 61% against Claude Sonnet 4.6 alone, with no accuracy lost ([benchmark](https://github.com/grounded-ai/grounded_ai/blob/main/benchmarks/tool_call/bench.py)).

## Setup

**In Colab:** add your keys under Secrets (the key icon on the left) and give this notebook access: `ANTHROPIC_API_KEY` for the judge, and `TYPESAFE_API_KEY` (Jev) and `OPENAI_API_KEY` (OpenAI Decisions) for the hosted decision models you want to try. Sections skip the models you have no key for. A GPU runtime (Runtime > Change runtime type > T4 GPU, free) makes agent-decision-3b fast; on a CPU it is slower.

**Elsewhere:** `pip install "grounded-ai[jev]"` and set the same keys as environment variables.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q "grounded-ai[jev]>=2.1.0"
    from google.colab import userdata

    for key in ("ANTHROPIC_API_KEY", "TYPESAFE_API_KEY", "OPENAI_API_KEY"):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:  # not set, or the notebook has no access to it
            pass
print("keys:", [key for key in ("ANTHROPIC_API_KEY", "TYPESAFE_API_KEY", "OPENAI_API_KEY") if os.getenv(key)])

keys: ['ANTHROPIC_API_KEY', 'TYPESAFE_API_KEY', 'OPENAI_API_KEY']


**agent-decision-3b**, our open decision model, needs no key: in Colab this cell serves it next to the notebook at `/v1/systemone` (the first run downloads about 7 GB). Elsewhere, serve it with `python serve/decision_server.py` from the [agent-decision repo](https://github.com/grounded-ai/agent-decision) (a CPU works, more slowly) and set `AGENT_DECISION_URL=http://127.0.0.1:8765`.

In [2]:
import json
import subprocess
import time
import urllib.request

if IN_COLAB and not os.getenv("AGENT_DECISION_URL"):
    subprocess.run(["git", "clone", "-q", "https://github.com/grounded-ai/agent-decision", "/content/agent-decision"])
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=5.10.2", "peft>=0.18"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])  # Colab's torchao is too old for transformers 5
    server = subprocess.Popen([sys.executable, "/content/agent-decision/serve/decision_server.py"],
                              stdout=open("/content/agent_decision.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(180):
        try:
            health = json.load(urllib.request.urlopen("http://127.0.0.1:8765/health", timeout=5))
            os.environ["AGENT_DECISION_URL"] = "http://127.0.0.1:8765"
            print("agent-decision-3b is up on", health["device"])
            break
        except Exception:
            if server.poll() is not None:
                print(open("/content/agent_decision.log").read()[-2000:])
                break
            time.sleep(10)

agent-decision-3b is up on cuda


## Build a cascade

A decision model, a judge, and the confidence below which the judge takes over. Any model string `Evaluator` accepts works for either.

In [3]:
from grounded_ai import CascadeEvaluator
from grounded_ai.backends.jev import HALLUCINATION, ChoiceQuestion, JevInput
from grounded_ai.schemas import EvaluationError

DECISION_MODELS = {
    "jev/jev-latest": ({}, "TYPESAFE_API_KEY"),
    "openai-decisions/gpt-6-luna": ({}, "OPENAI_API_KEY"),
}
if os.getenv("AGENT_DECISION_URL"):  # set by the cell above, or by you
    DECISION_MODELS["systemone/agent-decision-3b"] = ({"base_url": os.environ["AGENT_DECISION_URL"]}, None)
available = [m for m, (_, key) in DECISION_MODELS.items() if key is None or os.getenv(key)]
print("decision models:", available)

cascade = CascadeEvaluator(
    decision_model=available[0],
    decision_model_kwargs=DECISION_MODELS[available[0]][0],
    judge="anthropic/claude-haiku-4-5",
    min_confidence=0.7,
)

decision models: ['jev/jev-latest', 'openai-decisions/gpt-6-luna', 'systemone/agent-decision-3b']


## Read a result

`result.answers` has one answer per question. Where the decision model was sure, it is the decision model's own answer, with probabilities and `confidence`. Where it was not, it is a `JudgedAnswer` with the judge's pick and reasoning (an LLM measures no probabilities, so it has none). `result.escalated` names the questions that went to the judge, and `result.decision` keeps the decision model's answers to all of them.

Four hallucination checks. Clear cases like these are what the decision model settles alone: none of them reaches the judge.

In [4]:
context = (
    "Apollo 11 landed on the Moon in 1969. Neil Armstrong and Buzz Aldrin walked on the surface "
    "while Michael Collins stayed in orbit."
)
responses = [
    "Michael Collins stayed in orbit.",
    "Buzz Aldrin stayed in orbit while Collins walked on the Moon.",
    "Apollo 11 landed on the Moon in 1972.",
    "Armstrong was the only astronaut on the mission to walk on the Moon.",
]


def show(result, label):
    if isinstance(result, EvaluationError):
        raise RuntimeError(f"{result.error_code}: {result.message}")
    answer = result.answers["verdict"]
    first = result.decision.answers["verdict"]
    if "verdict" in result.judged:
        print(f"judge     {answer.answer:13} (decision model said {first.choice}, confidence {first.confidence:.2f})  {label}")
    else:
        print(f"decision  {answer.choice:13} confidence {answer.confidence:.2f}  {label}")


for response in responses:
    result = cascade.evaluate(JevInput(state={"context": context, "response": response},
                                       questions={"verdict": HALLUCINATION}))
    show(result, response)

decision  faithful      confidence 1.00  Michael Collins stayed in orbit.
decision  hallucination confidence 1.00  Buzz Aldrin stayed in orbit while Collins walked on the Moon.
decision  hallucination confidence 1.00  Apollo 11 landed on the Moon in 1972.


decision  hallucination confidence 1.00  Armstrong was the only astronaut on the mission to walk on the Moon.


A harder case, with several questions in one request: the decision model answers them all at once, and only the unsure ones go to the judge, together, in one call.

In [5]:
ticket = "Can you also cc my manager on the refund? Her email is in the thread."
result = cascade.evaluate(JevInput(
    state=ticket,
    questions={
        "action": ChoiceQuestion(
            instructions="What should the support agent do next?",
            criteria={"call_tool": "use a tool (email, refund system)", "respond": "reply to the customer",
                      "ask": "ask the customer for missing details"},
        ),
        "tone": ChoiceQuestion(instructions="Classify the customer's tone.",
                               criteria={"calm": "neutral or polite", "upset": "angry or frustrated"}),
    },
))
for name, answer in result.answers.items():
    who = "judge" if name in result.judged else "decision model"
    pick = answer.answer if name in result.judged else answer.choice
    print(f"{name:7} {pick:10} from the {who}")
print("escalated:", result.escalated)

action  call_tool  from the judge
tone    calm       from the judge
escalated: ['action', 'tone']


The judge's reasoning, for each question it answered:

In [6]:
for name in result.judged:
    print(f"{name}: {result.answers[name].reasoning}")
    print()

action: The customer is asking for their manager to be CC'd on a refund. They mention the manager's email is already in the thread, so the support agent has the information needed. The agent should take action by calling the appropriate tool (refund system, email system) to process the refund and CC the manager, rather than asking for more details or simply responding without action.
tone: The customer's message is polite and straightforward. They use 'also' and 'can you' in a courteous manner. There is no indication of frustration, anger, or upset tone. The request is presented calmly and cooperatively.


## Swap the decision model

Only the `decision_model` changes. Every decision model answers the same typed questions, and grounded-ai reads confidence with one definition for all of them ([TypeSafe's](https://docs.typesafe.ai/confidence)), so `min_confidence=0.7` means the same thing whichever you use.

agent-decision-3b is here when the setup cell started it (or you set `AGENT_DECISION_URL`). For a decision model on llama.cpp or Ollama, pass `host="llamacpp"` or `host="ollama"` in `decision_model_kwargs` instead of a `base_url`.

In [7]:
response = responses[3]
for model in available:
    swapped = CascadeEvaluator(decision_model=model, decision_model_kwargs=DECISION_MODELS[model][0],
                               judge="anthropic/claude-haiku-4-5", min_confidence=0.7)
    print(model)
    show(swapped.evaluate(JevInput(state={"context": context, "response": response},
                                   questions={"verdict": HALLUCINATION})), "")

jev/jev-latest
decision  hallucination confidence 1.00  


openai-decisions/gpt-6-luna


decision  hallucination confidence 0.98  
systemone/agent-decision-3b
decision  hallucination confidence 0.92  


## Pick a decision model and threshold on your own data

`compare()` asks each decision model every labelled item and reports, for each threshold, how much it settles alone (`settled`), how often those answers are right (`right`), and with a judge, the cascade's accuracy. With `target_accuracy` it recommends the lowest threshold that meets it. The judge is asked once per item, and only about items some model would send it.

An item is a state, its questions, and the right answer to each (the option for a choice, `True`/`False` for a yes/no, the level index for a score). The hallucination checks above are too easy to tell models apart, so here are 30 rows of agent decisions from [`agent-decision-eval`](https://huggingface.co/datasets/grounded-ai/agent-decision-eval), six from each question family: should a proposed tool call run, is an agent step on track, call or ask or decline, call a tool or not, is a reply grounded. agent-decision was trained on data like this (a held-out split of the same sources), so it has a home advantage; use a few hundred items of your own traffic to decide.

In [8]:
import collections
import json
import random

from huggingface_hub import hf_hub_download

from grounded_ai import Evaluator, compare

path = hf_hub_download("grounded-ai/agent-decision-eval", "test.jsonl", repo_type="dataset")
rows = [json.loads(line) for line in open(path)]
by_family = collections.defaultdict(list)
for row in rows:
    families = set(row["families"].values())
    by_family["tool_call" if "tool_call_ok" in families else families.pop()].append(row)
rng = random.Random(7)
items = [{"state": row["state"], "questions": row["questions"], "labels": row["gold"]}
         for family in sorted(by_family) for row in rng.sample(by_family[family], 6)]
print(len(items), "items,", sum(len(item["questions"]) for item in items), "questions")

models = {name: Evaluator(name, **DECISION_MODELS[name][0]) for name in available}
report = compare(items, models, judge="anthropic/claude-haiku-4-5",
                 thresholds=(0.5, 0.7, 0.9), target_accuracy=0.9)
print(report.table())

30 items, 32 questions


model                                          n accuracy    ECE  s/check  tokens  @0.5  settled/right/cascade  @0.7  settled/right/cascade  @0.9  settled/right/cascade
jev/jev-latest                                32    0.938  0.082    0.094    1569          88%/0.96/0.906          88%/0.96/0.906          66%/0.95/0.906
                                             act alone at >= 0.5 for 90% right
openai-decisions/gpt-6-luna                   32    0.938  0.085    0.273    1349          97%/0.94/0.938          91%/0.93/0.938          84%/0.93/0.938
                                             act alone at >= 0.5 for 90% right
systemone/agent-decision-3b                   32    0.969  0.072    0.940    1247          94%/1.00/1.000          84%/1.00/0.969          66%/1.00/0.969
                                             act alone at >= 0.5 for 90% right
judge: anthropic/claude-haiku-4-5, 0.909 right on the items it saw


Everything in the table is also on the report object (`report.models[name].thresholds`, `.recommended_threshold`, ...), and `report.model_dump_json()` saves it.

The same from the command line, with items as JSONL. `--save-records` and `--records` merge models that run on different machines, such as a GPU box serving agent-decision and a laptop with API keys:

```bash
grounded-ai compare items.jsonl --model jev/jev-latest --model openai-decisions/gpt-6-luna \
    --model systemone/agent-decision-3b@http://127.0.0.1:8765 --judge anthropic/claude-haiku-4-5 \
    --thresholds 0.5 0.7 0.9 --target-accuracy 0.9
```